# 1.-Liberías

In [17]:
import json 
from decimal import Decimal

import pandas as pd
from sqlalchemy import create_engine

from typing import List
import boto3
from pprint import pprint

# 2.-Configuraciones Generales

In [6]:
PG_USER = "acatlan"
PG_PASS = "8g4izJPX"
PG_DB = "db_acatlan"
PG_HOST = "172.18.0.5"
PG_PORT = "5432"
SCHEMA = "public"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)


# 3.- Creación DataFrame

In [7]:
query_database = """

SELECT c.customer_id ,
	   o.order_id,
	   oi.order_item_id,
	   p.product_category_name ,
	   oi.price 
FROM customers c 
INNER JOIN orders o 
	ON c.customer_id  = o.customer_id 
INNER JOIN order_items oi 
	ON oi.order_id  = o.order_id 
INNER JOIN products p 
	ON p.product_id = oi.product_id 
"""

In [8]:
df_postgres = pd.read_sql(query_database,engine)

In [9]:
df_postgres.head()

,customer_id,order_id,order_item_id,product_category_name,price
0,f6dd3ec061db4e3987629fe6b26e5cce,00018f77f2f0320c557190d7a144bdd3,1,pet_shop,239.9
1,6489ae5e4333f3693df5ad4372dab6d3,000229ec398224ef6ca0657da4fc703e,1,moveis_decoracao,199.0
2,58dbd0b2d70206bf40e62cd34e84d795,00042b26cf59d7ce69dfabb4e55b4fd9,1,ferramentas_jardim,199.9
3,816cbea969fe5b689b39cfc97a506742,00048cc3ae777c65dbb7d2a0634bc1ea,1,utilidades_domesticas,21.9
4,32e2e6ab09e778d99bf2e0ecd4898718,00054e8431b9d7675808bcb819fb4a32,1,telefonia,19.9


In [10]:
df_postgres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 5 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   customer_id            112650 non-null  object 
 1   order_id               112650 non-null  object 
 2   order_item_id          112650 non-null  int64  
 3   product_category_name  111047 non-null  object 
 4   price                  112650 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.3+ MB


# 4.-Traduccion SQL to NO SQL

In [11]:
item_to_dynamo_db : List = []

for order_id,sub_df in df_postgres.groupby("order_id"):

    item_to_dynamo_db.append(
        {
            "customer_id" : sub_df["customer_id"],
            "order_id" : sub_df["order_id"],
            "items": sub_df[["order_item_id","price"]].to_dict("records")
        }     
    )
    

In [31]:
pprint(
    item_to_dynamo_db[0:3],
    sort_dicts=True, 
)

[{'customer_id': 56291    3ce436f183e68e07877b285a838db11a
Name: customer_id, dtype: object,
  'items': [{'order_item_id': 1, 'price': 58.9}],
  'order_id': 56291    00010242fe8c5a6d1ba2dd792cb16214
Name: order_id, dtype: object},
 {'customer_id': 0    f6dd3ec061db4e3987629fe6b26e5cce
Name: customer_id, dtype: object,
  'items': [{'order_item_id': 1, 'price': 239.9}],
  'order_id': 0    00018f77f2f0320c557190d7a144bdd3
Name: order_id, dtype: object},
 {'customer_id': 1    6489ae5e4333f3693df5ad4372dab6d3
Name: customer_id, dtype: object,
  'items': [{'order_item_id': 1, 'price': 199.0}],
  'order_id': 1    000229ec398224ef6ca0657da4fc703e
Name: order_id, dtype: object}]


# 5.-Conexion a DynamoDB

In [32]:
dynamo_db = boto3.resource(
       "dynamodb",
    endpoint_url="http://ministack:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test" 
)

# 6.-Creacion del Artefacto : Tabla (dynamodb)

In [33]:
try:
    table = dynamo_db.create_table(
        TableName="tabla_ejemplo_dynamodb",
        KeySchema=[{"AttributeName": "customer_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "customer_id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST"
    )
except dynamo_db.meta.client.exceptions.ResourceInUseException:
    tabla = dynamo.Table("tabla_ejemplo_dynamodb_2")

# 7.- Inserción Base De Datos

In [35]:
for unit_item in item_to_dynamo_db:
    # Dynamo no acepta float ni fechas: esto los pasa a Decimal y texto
    item = (
            json.loads(
                        json.dumps(unit_item, default=str), parse_float=Decimal
                     )
           )
    table.put_item(Item=item)